# LedgerLens: transaction analytics walkthrough

This notebook uses the same validated functions as the dashboard. The included data is wholly fictional. No trained fraud classifier or commercial impact is claimed. See `docs/METHODOLOGY.md` for metric definitions.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'utils').is_dir():
    raise RuntimeError('Open this notebook from the repository root.')
sys.path.insert(0, str(ROOT))

from utils.data_loader import load_data
from utils.analytics import kpis, monthly_summary, risk_by_group, segment_summary, opportunity_lists
from utils.segmentation import customer_profile
from utils.visualizations import monthly_chart, segment_chart
from scripts.run_sql import run_queries

bundle = load_data(ROOT / 'data' / 'demo')
transactions = bundle.transactions
bundle.quality

## 1. Establish the denominator

Net value includes error-free purchases minus refunds. Fraud rates exclude unknown labels; coverage is reported separately. Transaction value is not bank revenue.

In [ ]:
kpis(transactions)

In [ ]:
monthly_chart(monthly_summary(transactions)).show()

## 2. Customer segments

Use positive, error-free purchases in the trailing 365-day window. Average ranks preserve ties; low-recency customers are checked for risk before loyalty. The full registry includes customers without purchase history.

In [ ]:
profile = customer_profile(transactions, bundle.users, bundle.cards, window_days=365)
segment_summary(profile)

In [ ]:
segment_chart(segment_summary(profile)).show()

## 3. Investigate observed fraud patterns

Use known labels only and require at least 30 observations per group. Wilson intervals assume independent observations; repeated customers and selective labeling can make them optimistic.

In [ ]:
risk_by_group(transactions, 'merchant_category', min_labeled=30)

## 4. Convert a finding into a testable hypothesis

Candidate lists are starting points for controlled experiments, not credit decisions or demonstrated uplift. Existing Credit card owners are excluded from discovery using the complete card registry.

In [ ]:
retention, cross_sell = opportunity_lists(profile)
print(f'Retention candidates: {len(retention)}; discovery candidates: {len(cross_sell)}')
cross_sell[['client_id', 'Frequency', 'Monetary', 'has_credit_card']].head(10)

## 5. Reproduce in SQL

The SQL runner creates an in-memory SQLite database. Automated tests compare monthly value, fraud rates and raw RFM metrics against Python. SQL cross-sell uses percent-rank, whose boundary behavior can differ from interpolated quantiles.

In [ ]:
sql_results = run_queries(bundle)
sql_results['01_monthly_value'].head()

## 6. Findings, assumptions and next experiment

Read `docs/DEMO_FINDINGS.md` for generated numerical findings. A real reactivation pilot would randomize eligible customers, predefine a 30-day reactivation metric and track incentive cost. No causal effect can be inferred from this notebook alone.

The original course notebook is retained in Git history. This walkthrough removes duplicated preprocessing so the notebook and application use the same analytical contract.